# Лабораторная работа 2R
# K-Means Clustering

### 4.1a        (K-means)  You have been asked to cluster all 50 U.S. states, including Washington D.C. and Puerto Rico, by mean household income and mean household electricity usage (both rounded by the integer).  You have decided to use a k-means clustering algorithm.

### Access the census data in the training2 database.  Create a table with three columns: state, mean household income, and mean electricity usage.  Note: If the housing_nodupes table isn’t present in the database, repeat the steps from Lab Exercise 1.3 Database Environment – Census Data to recreate it.

## Загружаем данные из предоставленного .Rdata файла

In [ ]:
load("resources/2-lab/income_elec_state.Rdata")

### Посмотрим, какие объекты находятся в файле данных

In [ ]:
ls()

### Посмотрим, что находится в ```income_elce_state```

In [ ]:
head(income_elec_state)

## a.      Cluster the data and plot all 52 data points, along with the centroids.  Mark all data points and centroids belonging to a given cluster with their own color.  Here, let k=10.

In [ ]:
library(ggplot2)

- ```Кластеризация``` - задача машинного обучения, которая подразумевает классификацию данных в n групп так, чтобы элементы одной и той же группы были максимально возможно похожи друг на друга
- ```Центроид``` - "центр" кластера, который вычисляется путем нахождения средне-арифметического координат элементов кластера. Можно сказать, ```центроид``` - усредненный представитель группы
- ```kmeans()``` - алгоритм, который случайно выбирает centers случайных точек, которые становятся начальными центроидами, после каждая точка "приписывается" к тому центроиду, к которому оказалась ближе остальных. Далее вычисляются для каждой группы новые центроиды, после чего цикл перепроверяет, нет ли "более близкого центроида из внешней группы для элемента". Когда элементы перестают "переходить" между группами, цикл заканчивается
- ```scale()``` - выполняет стандартизацию данных: центрирование ( Вычет из каждого элемента ср ариф столбца для смещения центра распределения в ноль ) и масштабирование ( Результирующую величину делим на стандартное отклонение столбца )
- ```Стандартное отклонение``` - корень дисперсии $$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \mu)^2}, \quad  где \ \ \mu \ \ средне-арифметическое$$
- ```sweeep()``` - предназначена для поэлементного выполнения какой-либо операции, в нашем случае $$\text{income\_elec\_state} = (\text{scaled\_income\_elec\_state} \times \sigma) + \mu$$
  можно сказать, что мы делаем $$ scale()^{-1}$$


In [ ]:
set.seed(42)

scaled_income_elec_state <- scale(income_elec_state[, c("income", "elec")])
kmeans_df <- kmeans(scaled_income_elec_state, centers = 10)

# По стандарту всего 8 цветов, и потому для соблюдения правила "1 цвет - 1 кластер"
# я создаю вектор, содержащий 10 цветов для моих k=10 кластеров
colors_for_plot <- c(
    "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", 
    "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ac"
)

plot(income_elec_state$income, income_elec_state$elec, 
     col = colors_for_plot[kmeans_df$cluster], 
     xlab = "Income", ylab = "Electricity",
     main = "K-means clustering"
)


centers <- sweep(sweep(kmeans_df$centers, 2, attr(scaled_income_elec_state, "scaled:scale"), "*"), 2, attr(scaled_income_elec_state, "scaled:center"), "+")
points(centers[, "income"], centers[, "elec"], 
       col = colors_for_plot,
       pch = 8,
       cex = 2, # во сколько раз увеличить размер символа
       lwd = 2 # толщина линий
)

Если опираться на тест, то хватит более простого графика

In [ ]:
set.seed(42)

kmeans_df <- kmeans(income_elec_state, centers = 10)

plot(income_elec_state$income, income_elec_state$elec, 
     col = kmeans_df$cluster, 
     xlab = "Income", ylab = "Electricity",
     main = "K-means clustering"
)

points(kmeans_df$centers[, "income"], kmeans_df$centers[, "elec"], 
       col = 1:10,   
       pch = 8
)

### b.      Repeat step (a) several times.  What can change each time you cluster the data?  Why?  How do you prevent these changes from occurring?

При повторном построении графика берутся другие случайные k начальных центроидов, в связи с чем алгоритм разбиения на кластеры не детерминирован и может давать отличный от предыдущего результат при новом запуске

In [ ]:
par(mfrow = c(2, 2))

for (i in 1:4) {
    kmeans_df <- kmeans(income_elec_state, centers = 10)
    
    plot(income_elec_state$income, income_elec_state$elec, 
         col = kmeans_df$cluster, 
         xlab = "Income", ylab = "Electricity",
         main = "K-means clustering"
    )
    
    points(kmeans_df$centers[, "income"], kmeans_df$centers[, "elec"], 
           col = 1:10,   
           pch = 8
    )
}

Для предотвращения "случайного" поведения следует прибегать к ```set.seed()```, посольку в таком случае алгоритм будет начинать с одного и того же семечка и, следовательно, выбирать одни и те же k начальных центроидов, поскольку остальные этапы алгоритма детермированы, то и результат будет детерминирован

In [ ]:
par(mfrow = c(2, 2))

for (i in 1:4) {
    set.seed(67)
    kmeans_df <- kmeans(income_elec_state, centers = 10)
    
    plot(income_elec_state$income, income_elec_state$elec, 
         col = kmeans_df$cluster, 
         xlab = "Income", ylab = "Electricity",
         main = "K-means clustering"
    )
    
    points(kmeans_df$centers[, "income"], kmeans_df$centers[, "elec"], 
           col = 1:10,   
           pch = 8
    )
}
par(mfrow = c(1, 1))

## c.       Once you’ve accounted for the issues in the previous step, determine a reasonable value of k.  Why would you suggest this value of k?

### Для оценки k воспользуемся методом силуэта
[Ссылка на источник ( Статья про коэффициент силуэта )](https://www.datacamp.com/tutorial/silhouette-score?dc_referrer=https%3A%2F%2Fgemini.google.com%2F)

In [ ]:
# устанавливаем библиотеку cluster
if (!requireNamespace("cluster", quietly = TRUE)) {
  install.packages("cluster")
}

- ```sapply(v, f)``` - последовательно применяет функцию ```f``` к каждому элементу вектора ```v```

### Коэффициент силуэта помогает оценить плотность и обособленность элементов

Сначала для каждого элемента датасета считается
$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$
$$a_i \ - \ средне-арифметическое \ расстояний \ между \ элементом_i \ и \ остальными \ элементами \ кластера$$
$$b_i \ - \ минимальное \ из \ средних \ расстояний \ между \ элементом_i \ и \ всеми \ элементами \ другого \ кластера$$
Затем мы находим средне-арифметическое коэффициентов силуэта каждого элемента - это и будет нашим "коэффициентом силуэта разбиения для k"

In [ ]:
library(cluster)

k_values <- 2:15

silhouette <- sapply(k_values, function(k) {
    # nstart - сколько произвести независимых запусков
    # после отработки kmeans() вернет вариант с наименьшей внутрикластерной дисперсией 
    km <- kmeans(income_elec_state, centers = k, nstart = 25)
    # Рассчитывает для каждого элемента его коэфф силуэта
    sil <- silhouette(km$cluster, dist(income_elec_state))
  
    mean(sil[, "sil_width"])
})

plot(k_values, silhouette, 
     type = "b", 
     pch = 19, 
     xlab = "k", 
     ylab = "silhouette", 
     main = "Silhouette Score")

Опираясь на график ```Silhouette Score``` понятно, что k = 8 является оптимальным, построим график с k = 8

In [ ]:
set.seed(18)
kmeans_df <- kmeans(income_elec_state, centers = 8)
    
plot(income_elec_state$income, income_elec_state$elec, 
    col = kmeans_df$cluster, 
    xlab = "Income", ylab = "Electricity",
    main = "K-means clustering", pch = 19
)
    
points(kmeans_df$centers[, "income"], kmeans_df$centers[, "elec"], 
    col = 1:10,
    pch = 8, cex = 2, lwd = 2
)

## d.      Convert the mean household income and mean electricity usage to a log10 scale and cluster this transformed dataset.  How has the clustering changed?  Why?